# Undercover Agent City — Sequential Curriculum Training (4B/7B)

**Sequential curriculum**: Phase 1 (tutorial) → Phase 2 (first_contact) → Phase 3 (earn_trust)  
Evaluates after **each** phase to track progression and detect forgetting.  
Auto-detects GPU: **Qwen3-4B** on T4, **Qwen3-7B** on L4/A100.  
LoRA rank **r=32** (higher capacity to prevent catastrophic forgetting across phases).

In [ ]:
%%capture
# === CELL 1: Official Unsloth install (from unslothai/notebooks) ===
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    pass

In [ ]:
%%capture
# === CELL 2: Colab Extra Install (exact copy from official Unsloth notebooks) ===
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
# Kill torchcodec if pulled in (causes FFmpeg crashes)
!pip uninstall torchcodec -y 2>/dev/null
!pip uninstall sentence-transformers -y 2>/dev/null
# Install our environment
!pip install openenv-core -q

## RESTART RUNTIME NOW

Go to **Runtime -> Restart runtime** (or Ctrl+M then period).

Then **skip to Cell 4** below. Do NOT re-run Cells 1-2.

In [ ]:
# === CELL 4: Setup after restart ===
import os, sys
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# GPU check
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Unzip and install environment
if os.path.exists("/content/undercover_agent_city.zip"):
    !cd /content && unzip -oq undercover_agent_city.zip
    !pip install -e /content/undercover_agent_city/ -q 2>/dev/null
sys.path.insert(0, "/content")

# Verify environment works
from undercover_agent_city.models import CityAction
from undercover_agent_city.server.city_environment import CityEnvironment
env = CityEnvironment()
obs = env.reset(task="tutorial_persona")
print(f"Env OK! {len(obs.available_actions)} actions, nearby: {[n['name'] for n in obs.nearby_npcs]}")
del env, obs

In [ ]:
# === CELL 5: Generate THREE datasets ===
import json, random, re
from datasets import Dataset

def collect_observations(task, n_episodes=200, seed_start=0):
    """Collect observations by running random episodes."""
    env = CityEnvironment()
    raw = []
    for ep in range(n_episodes):
        obs = env.reset(task=task, seed=seed_start + ep)
        rng = random.Random(seed_start + ep + 1000)
        for step in range(obs.max_turns):
            if obs.done:
                break
            raw.append({
                "prompt_text": obs.message,
                "num_actions": len(obs.available_actions),
                "actions_json": json.dumps(obs.available_actions),
                "task": task,
                "seed": seed_start + ep,
                "step_idx": step,
            })
            obs = env.step(CityAction(action=rng.randint(1, len(obs.available_actions))))
    return raw

print("Collecting tutorial_persona observations (200 episodes)...")
raw_tutorial = collect_observations("tutorial_persona", n_episodes=200, seed_start=0)
print(f"  tutorial_persona: {len(raw_tutorial)} observations")

print("Collecting first_contact observations (200 episodes)...")
raw_first_contact = collect_observations("first_contact", n_episodes=200, seed_start=1000)
print(f"  first_contact: {len(raw_first_contact)} observations")

print("Collecting earn_trust observations (200 episodes)...")
raw_earn_trust = collect_observations("earn_trust", n_episodes=200, seed_start=2000)
print(f"  earn_trust: {len(raw_earn_trust)} observations")

print(f"\nTotal: {len(raw_tutorial) + len(raw_first_contact) + len(raw_earn_trust)} observations across 3 tasks")

In [ ]:
# === CELL 6: Format datasets + Define system prompt ===

# NOTE on /no_think: We put it in the system prompt so the model knows not to
# reason internally. But the CRITICAL fix is passing enable_thinking=False via
# chat_template_kwargs in GRPOConfig -- otherwise the Qwen3 chat template
# prepends <think>\n to the generation, the model fills it with thinking tokens,
# those get truncated by max_completion_length, mask_truncated_completions=True
# masks them out, and ALL generations end up identical => reward_std = 0.

SYSTEM_PROMPT = (
    "You are an undercover agent infiltrating a criminal gang in a city. /no_think\n"
    "Pick the BEST action number. \n"
    "TOUGH with gang, FORMAL with police, CASUAL with civilians. \n"
    "Keep your cover story CONSISTENT — never contradict what you told same-faction NPCs.\n"
    "Output ONLY a single number. Nothing else."
)

def format_row(obs_list):
    """Convert raw observations into HF Dataset with chat-format prompts."""
    return Dataset.from_list([{
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": o["prompt_text"]},
        ],
        **{k: o[k] for k in ["num_actions", "actions_json", "task", "seed", "step_idx"]},
    } for o in obs_list])

tutorial_dataset = format_row(raw_tutorial)
first_contact_dataset = format_row(raw_first_contact)
earn_trust_dataset = format_row(raw_earn_trust)

print(f"Tutorial dataset:       {len(tutorial_dataset)} rows")
print(f"First contact dataset:  {len(first_contact_dataset)} rows")
print(f"Earn trust dataset:     {len(earn_trust_dataset)} rows")

In [ ]:
# === CELL 7: Load model — AUTO-DETECT GPU ===
from unsloth import FastLanguageModel
import torch

# Auto-detect GPU for model selection
gpu_name = torch.cuda.get_device_name(0)
is_t4 = "T4" in gpu_name
MODEL_NAME = "unsloth/Qwen3-4B" if is_t4 else "unsloth/Qwen3-7B"
NUM_GENERATIONS = 2 if is_t4 else 4  # T4 has less VRAM

max_seq_length = 512
lora_rank = 32  # Higher for sequential to prevent forgetting

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9,
)
model = FastLanguageModel.get_peft_model(
    model, r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,  # 64
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"Model: {MODEL_NAME} | GPU: {gpu_name} | Generations: {NUM_GENERATIONS}")
print(f"LoRA: r={lora_rank}, alpha={lora_rank*2}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# === CELL 8: Define reward functions ===
import random as stdlib_random

def parse_action_number(text, max_actions):
    """Extract action number from model output. Strips any XML tags like <think>."""
    text = re.sub(r"<[^>]+>.*?</[^>]+>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"<[^>]+>", "", text).strip()
    digits = "".join(c for c in text if c.isdigit())
    if digits:
        n = int(digits[0])
        if 1 <= n <= max_actions:
            return n
    return None

def format_reward(completions, **kwargs):
    """1.0 for valid action number, -1.0 otherwise."""
    na = kwargs.get("num_actions", [8] * len(completions))
    return [
        1.0 if parse_action_number(
            c[0]["content"] if isinstance(c, list) else c, int(na[i])
        ) is not None else -1.0
        for i, c in enumerate(completions)
    ]

def env_reward(completions, num_actions, actions_json, task, seed, step_idx, **kwargs):
    """Replay environment to the exact step, execute chosen action, return scaled reward."""
    scores = []
    for i, c in enumerate(completions):
        text = c[0]["content"] if isinstance(c, list) else c
        a = parse_action_number(text, int(num_actions[i]))
        if a is None:
            scores.append(-2.0)
            continue
        try:
            e = CityEnvironment()
            o = e.reset(task=str(task[i]), seed=int(seed[i]))
            t = int(step_idx[i])
            rng = stdlib_random.Random(int(seed[i]) + 1000)
            for _ in range(t):
                if o.done:
                    break
                o = e.step(CityAction(action=rng.randint(1, len(o.available_actions))))
            if o.done:
                scores.append(-1.0)
                continue
            r = e.step(CityAction(action=a))
            scores.append((r.reward or 0.0) * 8.0)
        except Exception:
            scores.append(-2.0)
    return scores

# Quick sanity test
e = CityEnvironment()
o = e.reset(task="tutorial_persona", seed=0)
print("Reward sanity test (tutorial_persona seed=0):")
for a in o.available_actions:
    e2 = CityEnvironment()
    e2.reset(task="tutorial_persona", seed=0)
    r = e2.step(CityAction(action=a["index"]))
    print(f"  {a['description'][:50]}: reward={r.reward:.3f} (x8={r.reward*8:.3f})")
print("Reward functions OK!")

In [ ]:
# === CELL 9: Evaluation functions + Pre-training baseline ===

def evaluate_model(model, tokenizer, task, n_episodes=30, seed_start=10000):
    """Evaluate trained model on a task. Returns dict with persona_acc, grade, cover_intact."""
    env = CityEnvironment()
    correct = total_talks = 0
    grades = []
    cover_blown = 0

    for ep in range(n_episodes):
        obs = env.reset(task=task, seed=seed_start + ep)
        for _ in range(obs.max_turns):
            if obs.done:
                break
            msgs = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": obs.message},
            ]
            prompt = tokenizer.apply_chat_template(
                msgs, add_generation_prompt=True, tokenize=False,
                enable_thinking=False,
            )
            inp = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inp, max_new_tokens=8,
                    temperature=0.1, do_sample=True,
                )
            resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
            action = parse_action_number(resp, len(obs.available_actions)) or 1
            obs = env.step(CityAction(action=action))
            if obs.metadata and obs.metadata.get("was_talk_action"):
                total_talks += 1
                if obs.metadata.get("persona_correct"):
                    correct += 1
            if obs.metadata and obs.metadata.get("cover_blown"):
                cover_blown += 1
        grades.append(env.grade_episode())

    return {
        "persona_acc": correct / max(1, total_talks),
        "grade": sum(grades) / len(grades),
        "cover_intact": 1 - (cover_blown / n_episodes),
        "n_episodes": n_episodes,
    }

def evaluate_random(task, n_episodes=30, seed_start=10000):
    """Random baseline for comparison."""
    env = CityEnvironment()
    correct = total_talks = 0
    grades = []
    cover_blown = 0

    for ep in range(n_episodes):
        obs = env.reset(task=task, seed=seed_start + ep)
        rng = stdlib_random.Random(seed_start + ep + 5000)
        for _ in range(obs.max_turns):
            if obs.done:
                break
            obs = env.step(CityAction(action=rng.randint(1, len(obs.available_actions))))
            if obs.metadata and obs.metadata.get("was_talk_action"):
                total_talks += 1
                if obs.metadata.get("persona_correct"):
                    correct += 1
            if obs.metadata and obs.metadata.get("cover_blown"):
                cover_blown += 1
        grades.append(env.grade_episode())

    return {
        "persona_acc": correct / max(1, total_talks),
        "grade": sum(grades) / len(grades),
        "cover_intact": 1 - (cover_blown / n_episodes),
        "n_episodes": n_episodes,
    }

# Evaluate BEFORE any training
print("=" * 60)
print("BASELINE EVALUATION (before training)")
print("=" * 60)

baselines = {}
EVAL_TASKS = [("tutorial_persona", 30), ("first_contact", 10), ("earn_trust", 5)]

# Random baselines
print("\nCollecting random baselines...")
baselines["tutorial_persona_random"] = evaluate_random("tutorial_persona", 30)
baselines["first_contact_random"] = evaluate_random("first_contact", 10)
baselines["earn_trust_random"] = evaluate_random("earn_trust", 5)

# Untrained model baselines
print("Evaluating untrained model...")
baselines["tutorial_persona_untrained"] = evaluate_model(model, tokenizer, "tutorial_persona", 30)
baselines["first_contact_untrained"] = evaluate_model(model, tokenizer, "first_contact", 10)
baselines["earn_trust_untrained"] = evaluate_model(model, tokenizer, "earn_trust", 5)

for task, n in EVAL_TASKS:
    print(f"  {task}: random persona={baselines[f'{task}_random']['persona_acc']:.0%}  grade={baselines[f'{task}_random']['grade']:.3f}")
    print(f"  {task}: untrained persona={baselines[f'{task}_untrained']['persona_acc']:.0%}  grade={baselines[f'{task}_untrained']['grade']:.3f}")

print("\nBaseline collection complete.")

In [ ]:
# === CELL 10: Phase 1 — Train on tutorial_persona (200 steps) ===
from trl import GRPOConfig, GRPOTrainer

# ============================================================================
# FIX for reward_std = 0 (all generations identical):
#
# ROOT CAUSE: Without chat_template_kwargs={"enable_thinking": False}, TRL's
# GRPOTrainer calls tokenizer.apply_chat_template() with enable_thinking=True
# (the Qwen3 default). This prepends "<think>\n" to every generation, and the
# model then fills those tokens with thinking content. With
# max_completion_length=12, the thinking content is immediately truncated, and
# mask_truncated_completions=True zeros out the entire completion. Result: all
# generations have identical (empty/masked) outputs => reward_std = 0 =>
# zero gradient => model learns NOTHING.
#
# FIXES APPLIED:
# 1. chat_template_kwargs={"enable_thinking": False}
# 2. max_completion_length increased 12 -> 32
# ============================================================================

def make_grpo_config(max_steps, output_dir):
    """Shared GRPO config factory for all phases."""
    return GRPOConfig(
        # === Generation sampling ===
        temperature=1.0,
        top_p=0.95,
        num_generations=NUM_GENERATIONS,
        max_prompt_length=512,   # Was 450 — slight increase for longer observations
        max_completion_length=32,  # Was 12 — too short with <think> overhead
        # === CRITICAL: disable Qwen3 thinking in chat template ===
        chat_template_kwargs={"enable_thinking": False},
        # === Training hyperparams ===
        learning_rate=5e-6,
        weight_decay=0.1,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        logging_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=max_steps,
        save_steps=max_steps,
        max_grad_norm=0.1,
        epsilon=0.2,
        epsilon_high=0.28,
        mask_truncated_completions=True,
        report_to="none",
        output_dir=output_dir,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    )

# Store all training logs across phases
all_train_logs = []

print("=" * 60)
print("PHASE 1: tutorial_persona (200 steps)")
print("=" * 60)
_cfg = make_grpo_config(200, "outputs_phase1")
print(f"  chat_template_kwargs: {_cfg.chat_template_kwargs}")
print(f"  max_completion_length: {_cfg.max_completion_length}")

trainer_p1 = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, env_reward],
    reward_weights=[1.0, 8.0],
    args=_cfg,
    train_dataset=tutorial_dataset,
)
trainer_p1.train()
phase1_logs = trainer_p1.state.log_history.copy()
all_train_logs.append(("Phase 1: tutorial", phase1_logs))
print("Phase 1 training complete!")

In [ ]:
# === CELL 11: Evaluate after Phase 1 ===
print("=" * 60)
print("EVALUATION AFTER PHASE 1 (tutorial_persona)")
print("=" * 60)

eval_after_p1 = {}
eval_after_p1["tutorial_persona"] = evaluate_model(model, tokenizer, "tutorial_persona", 30)
eval_after_p1["first_contact"] = evaluate_model(model, tokenizer, "first_contact", 10)
eval_after_p1["earn_trust"] = evaluate_model(model, tokenizer, "earn_trust", 5)

for task in ["tutorial_persona", "first_contact", "earn_trust"]:
    u = baselines[f"{task}_untrained"]
    p = eval_after_p1[task]
    print(f"  {task}: persona={p['persona_acc']:.0%} (was {u['persona_acc']:.0%})  grade={p['grade']:.3f} (was {u['grade']:.3f})")

In [ ]:
# === CELL 12: Phase 2 — Train on first_contact (200 steps) ===
print("=" * 60)
print("PHASE 2: first_contact (200 steps)")
print("=" * 60)

trainer_p2 = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, env_reward],
    reward_weights=[1.0, 8.0],
    args=make_grpo_config(200, "outputs_phase2"),
    train_dataset=first_contact_dataset,
)
trainer_p2.train()
phase2_logs = trainer_p2.state.log_history.copy()
all_train_logs.append(("Phase 2: first_contact", phase2_logs))
print("Phase 2 training complete!")

In [ ]:
# === CELL 13: Evaluate after Phase 2 ===
print("=" * 60)
print("EVALUATION AFTER PHASE 2 (first_contact)")
print("=" * 60)

eval_after_p2 = {}
eval_after_p2["tutorial_persona"] = evaluate_model(model, tokenizer, "tutorial_persona", 30)
eval_after_p2["first_contact"] = evaluate_model(model, tokenizer, "first_contact", 10)
eval_after_p2["earn_trust"] = evaluate_model(model, tokenizer, "earn_trust", 5)

for task in ["tutorial_persona", "first_contact", "earn_trust"]:
    p1 = eval_after_p1[task]
    p2 = eval_after_p2[task]
    print(f"  {task}: persona={p2['persona_acc']:.0%} (was {p1['persona_acc']:.0%} after P1)  grade={p2['grade']:.3f} (was {p1['grade']:.3f})")

# Check for catastrophic forgetting on tutorial
t_p1 = eval_after_p1["tutorial_persona"]["persona_acc"]
t_p2 = eval_after_p2["tutorial_persona"]["persona_acc"]
if t_p2 < t_p1 - 0.1:
    print(f"\n  WARNING: Tutorial persona dropped {t_p1:.0%} -> {t_p2:.0%} (forgetting detected!)")
else:
    print(f"\n  Tutorial retention OK: {t_p1:.0%} -> {t_p2:.0%}")

In [ ]:
# === CELL 14: Phase 3 — Train on earn_trust (100 steps) ===
print("=" * 60)
print("PHASE 3: earn_trust (100 steps)")
print("=" * 60)

trainer_p3 = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, env_reward],
    reward_weights=[1.0, 8.0],
    args=make_grpo_config(100, "outputs_phase3"),
    train_dataset=earn_trust_dataset,
)
trainer_p3.train()
phase3_logs = trainer_p3.state.log_history.copy()
all_train_logs.append(("Phase 3: earn_trust", phase3_logs))
print("Phase 3 training complete!")

In [ ]:
# === CELL 15: Evaluate after Phase 3 (FINAL comparison table) ===
print("=" * 60)
print("FINAL EVALUATION AFTER PHASE 3 (earn_trust)")
print("=" * 60)

eval_after_p3 = {}
eval_after_p3["tutorial_persona"] = evaluate_model(model, tokenizer, "tutorial_persona", 30)
eval_after_p3["first_contact"] = evaluate_model(model, tokenizer, "first_contact", 10)
eval_after_p3["earn_trust"] = evaluate_model(model, tokenizer, "earn_trust", 5)

for task in ["tutorial_persona", "first_contact", "earn_trust"]:
    p2 = eval_after_p2[task]
    p3 = eval_after_p3[task]
    print(f"  {task}: persona={p3['persona_acc']:.0%} (was {p2['persona_acc']:.0%} after P2)  grade={p3['grade']:.3f} (was {p2['grade']:.3f})")

# Full progression table
print("\n" + "=" * 100)
print("SEQUENTIAL CURRICULUM PROGRESSION")
print("=" * 100)
print(f"{'Task':<20} {'Metric':<10} {'Random':>8} {'Untrained':>10} {'After P1':>9} {'After P2':>9} {'After P3':>9} {'Delta':>8}")
print("-" * 100)
for task, _ in EVAL_TASKS:
    rp = baselines[f"{task}_random"]["persona_acc"]
    up = baselines[f"{task}_untrained"]["persona_acc"]
    p1p = eval_after_p1[task]["persona_acc"]
    p2p = eval_after_p2[task]["persona_acc"]
    p3p = eval_after_p3[task]["persona_acc"]
    print(f"{task:<20} {'Persona':<10} {rp:>7.0%} {up:>9.0%} {p1p:>8.0%} {p2p:>8.0%} {p3p:>8.0%} {p3p-up:>+7.0%}")
    rg = baselines[f"{task}_random"]["grade"]
    ug = baselines[f"{task}_untrained"]["grade"]
    p1g = eval_after_p1[task]["grade"]
    p2g = eval_after_p2[task]["grade"]
    p3g = eval_after_p3[task]["grade"]
    print(f"{'':<20} {'Grade':<10} {rg:>8.3f} {ug:>10.3f} {p1g:>9.3f} {p2g:>9.3f} {p3g:>9.3f} {p3g-ug:>+8.3f}")
    print()

In [ ]:
# === CELL 16: Generate plots ===
import matplotlib.pyplot as plt
import numpy as np
import os
os.makedirs("outputs_sequential/plots", exist_ok=True)

# --- Plot 1: Reward curve with phase transitions ---
fig, ax = plt.subplots(figsize=(14, 5))
global_step = 0
colors = ["#2196F3", "#FF9800", "#4CAF50"]
phase_boundaries = []

for idx, (phase_name, logs) in enumerate(all_train_logs):
    steps, rewards = [], []
    for entry in logs:
        if "reward" in entry:
            steps.append(global_step + entry["step"])
            rewards.append(entry["reward"])
    if steps:
        ax.plot(steps, rewards, alpha=0.3, lw=0.5, color=colors[idx])
        if len(rewards) > 10:
            w = 10
            smoothed = np.convolve(rewards, np.ones(w) / w, mode="valid")
            ax.plot(steps[:len(smoothed)], smoothed, color=colors[idx], lw=2, label=phase_name)
        else:
            ax.plot(steps, rewards, color=colors[idx], lw=2, label=phase_name)
        global_step = steps[-1]
        phase_boundaries.append(global_step)

# Draw phase transition lines
for i, boundary in enumerate(phase_boundaries[:-1]):
    ax.axvline(x=boundary, color="red", linestyle="--", alpha=0.7, lw=1.5)
    ax.text(boundary, ax.get_ylim()[1] * 0.95, f"  Phase {i+2} start",
            color="red", fontsize=9, va="top")

ax.set(xlabel="Training Step (cumulative)", ylabel="Reward",
       title=f"Undercover Agent City -- Sequential Curriculum GRPO ({MODEL_NAME.split('/')[-1]})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs_sequential/plots/reward_curve_sequential.png", dpi=150)
plt.show()
print("Saved: outputs_sequential/plots/reward_curve_sequential.png")

# --- Plot 2: Progression bar charts (per-phase eval) ---
tasks = ["tutorial_persona", "first_contact", "earn_trust"]
x = np.arange(len(tasks))
width = 0.15

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Persona accuracy progression
eval_stages = [
    ("Random", {t: baselines[f"{t}_random"]["persona_acc"] for t in tasks}),
    ("Untrained", {t: baselines[f"{t}_untrained"]["persona_acc"] for t in tasks}),
    ("After P1", {t: eval_after_p1[t]["persona_acc"] for t in tasks}),
    ("After P2", {t: eval_after_p2[t]["persona_acc"] for t in tasks}),
    ("After P3", {t: eval_after_p3[t]["persona_acc"] for t in tasks}),
]
for i, (label, data) in enumerate(eval_stages):
    axes[0].bar(x + i * width, [data[t] for t in tasks], width, label=label)
axes[0].set(ylabel="Persona Accuracy", title="Persona Accuracy Progression")
axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels(tasks, rotation=15)
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.15)

# Grade progression
eval_stages_grade = [
    ("Random", {t: baselines[f"{t}_random"]["grade"] for t in tasks}),
    ("Untrained", {t: baselines[f"{t}_untrained"]["grade"] for t in tasks}),
    ("After P1", {t: eval_after_p1[t]["grade"] for t in tasks}),
    ("After P2", {t: eval_after_p2[t]["grade"] for t in tasks}),
    ("After P3", {t: eval_after_p3[t]["grade"] for t in tasks}),
]
for i, (label, data) in enumerate(eval_stages_grade):
    axes[1].bar(x + i * width, [data[t] for t in tasks], width, label=label)
axes[1].set(ylabel="Grade (0-1)", title="Task Grade Progression")
axes[1].set_xticks(x + width * 2)
axes[1].set_xticklabels(tasks, rotation=15)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 1.15)

plt.suptitle(f"Sequential Curriculum Results ({MODEL_NAME.split('/')[-1]}, 500 total steps)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs_sequential/plots/eval_progression.png", dpi=150)
plt.show()
print("Saved: outputs_sequential/plots/eval_progression.png")

# --- Plot 3: Comparison table with GPT-4o-mini ---
gpt_baselines = {
    "tutorial_persona": {"persona_acc": 1.00, "grade": 1.000},
    "first_contact": {"persona_acc": 0.98, "grade": 0.992},
    "earn_trust": {"persona_acc": 0.67, "grade": 0.483},
}

fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")
table_data = [["Task", "Metric", "Random", "GPT-4o-mini", "Untrained", "After P1", "After P2", "After P3 (Final)", "Delta"]]
for task in tasks:
    rp = baselines[f"{task}_random"]["persona_acc"]
    up = baselines[f"{task}_untrained"]["persona_acc"]
    p1p = eval_after_p1[task]["persona_acc"]
    p2p = eval_after_p2[task]["persona_acc"]
    p3p = eval_after_p3[task]["persona_acc"]
    table_data.append([
        task, "Persona",
        f"{rp:.0%}", f"{gpt_baselines[task]['persona_acc']:.0%}",
        f"{up:.0%}", f"{p1p:.0%}", f"{p2p:.0%}", f"{p3p:.0%}", f"{p3p-up:+.0%}",
    ])
    rg = baselines[f"{task}_random"]["grade"]
    ug = baselines[f"{task}_untrained"]["grade"]
    p1g = eval_after_p1[task]["grade"]
    p2g = eval_after_p2[task]["grade"]
    p3g = eval_after_p3[task]["grade"]
    table_data.append([
        "", "Grade",
        f"{rg:.3f}", f"{gpt_baselines[task]['grade']:.3f}",
        f"{ug:.3f}", f"{p1g:.3f}", f"{p2g:.3f}", f"{p3g:.3f}", f"{p3g-ug:+.3f}",
    ])

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                 loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
plt.title("Sequential Curriculum: Full Progression Table", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("outputs_sequential/plots/comparison_table.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs_sequential/plots/comparison_table.png")

In [ ]:
# === CELL 17: Save and download ===
import shutil

# Save LoRA adapter
model.save_lora("undercover_agent_city_sequential_lora")

# Package everything
os.makedirs("outputs_sequential/model", exist_ok=True)
if os.path.exists("undercover_agent_city_sequential_lora"):
    shutil.copytree("undercover_agent_city_sequential_lora", "outputs_sequential/model/lora_adapter", dirs_exist_ok=True)

# Create results JSON
results = {
    "model": MODEL_NAME,
    "training_type": "sequential_curriculum",
    "lora_rank": 32,
    "lora_alpha": 64,
    "phases": {
        "phase1": {"task": "tutorial_persona", "steps": 200},
        "phase2": {"task": "first_contact", "steps": 200},
        "phase3": {"task": "earn_trust", "steps": 100},
    },
    "total_steps": 500,
    "baselines": baselines,
    "eval_after_p1": {k: v for k, v in eval_after_p1.items()},
    "eval_after_p2": {k: v for k, v in eval_after_p2.items()},
    "eval_after_p3": {k: v for k, v in eval_after_p3.items()},
    "gpt4o_mini_baselines": gpt_baselines,
}
with open("outputs_sequential/results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

# Zip for download
!zip -rq outputs_sequential.zip outputs_sequential/
try:
    from google.colab import files
    files.download("outputs_sequential.zip")
except Exception:
    print("Files saved to outputs_sequential/ directory")

print("\nDone! Artifacts saved:")
print("  outputs_sequential/plots/reward_curve_sequential.png")
print("  outputs_sequential/plots/eval_progression.png")
print("  outputs_sequential/plots/comparison_table.png")
print("  outputs_sequential/model/lora_adapter/")
print("  outputs_sequential/results.json")